In [8]:
!pip3 install fastparquet



  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 715.4/715.4 kB 9.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 19.0 MB/s eta 0:00:00a 0:00:01
Using cached fsspec-2026.4.0-py3-none-any.whl (203 kB)

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [7]:
!pip3 install pyarrow


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


 # **Feature Engineering**

In [1]:
import pandas as pd
df_clean = pd.read_parquet("give the path of the folder where you have unzipped the file/Drif_detection_ds/df_clean.parquet")
print(f"Loaded df_clean: {df_clean.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'give the path of the folder where you have unzipped the file/Drif_detection_ds/df_clean.parquet'

 ── SECTION 1: Time Based Features ──


In [ ]:
import pandas as pd
import numpy as np

df_feat = df_clean.copy()


# Already have: pickup_hour, pickup_dow, pickup_month

# Is rush hour? (morning 7-9am, evening 4-7pm)
df_feat["is_rush_hour"] = df_feat["pickup_hour"].isin(
    list(range(7, 10)) + list(range(16, 20))
).astype(int)

# Is night time? (10pm - 5am) — NYC nightlife pattern
df_feat["is_night"] = df_feat["pickup_hour"].isin(
    list(range(22, 24)) + list(range(0, 6))
).astype(int)

# Is weekend?
df_feat["is_weekend"] = (df_feat["pickup_dow"] >= 5).astype(int)

# Season (important for NYC weather effect on trips)
def get_season(month):
    if month in [12, 1, 2]: return 0   # Winter
    elif month in [3, 4, 5]: return 1  # Spring
    elif month in [6, 7, 8]: return 2  # Summer
    else: return 3                      # Fall

df_feat["season"] = df_feat["pickup_month"].apply(get_season)

print("Time features done")
print(df_feat[["is_rush_hour","is_night","is_weekend","season"]].value_counts().head(10))

Time features done
is_rush_hour  is_night  is_weekend  season
0             0         0           1         73249
                                    0         69737
1             0         0           1         63573
                                    0         60453
0             0         0           3         43835
                                    2         42550
1             0         0           3         38210
                                    2         35682
0             1         0           1         25706
              0         1           1         25093
Name: count, dtype: int64


In [ ]:
import requests

# Download official TLC zone lookup
zone_lookup_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_df = pd.read_csv(zone_lookup_url)
print(zone_df.head())
print(zone_df["Borough"].value_counts())

# Build proper zone mappings from official data
AIRPORT_ZONES  = set(zone_df[zone_df["Zone"].str.contains("Airport", na=False)]["LocationID"])
MANHATTAN_ZONES = set(zone_df[zone_df["Borough"] == "Manhattan"]["LocationID"])

print(f"\nAirport zones: {AIRPORT_ZONES}")
print(f"Manhattan zones count: {len(MANHATTAN_ZONES)}")

   LocationID        Borough                     Zone service_zone
0           1            EWR           Newark Airport          EWR
1           2         Queens              Jamaica Bay    Boro Zone
2           3          Bronx  Allerton/Pelham Gardens    Boro Zone
3           4      Manhattan            Alphabet City  Yellow Zone
4           5  Staten Island            Arden Heights    Boro Zone
Borough
Queens           69
Manhattan        69
Brooklyn         61
Bronx            43
Staten Island    20
EWR               1
Unknown           1
Name: count, dtype: int64

Airport zones: {1, 138, 132}
Manhattan zones count: 69


Section 2. SPATIAL FEATURES

In [ ]:


# NYC TLC zone categories — manually mapped from TLC zone lookup
# These are the 3 borough groups most relevant to trip behaviour
# MANHATTAN_ZONES = set(range(1, 70))       # approximate Manhattan zone IDs
# AIRPORT_ZONES   = {1, 132, 138}           # Newark, JFK, LaGuardia


def zone_type(zone_id):
    if zone_id in AIRPORT_ZONES:
        return "airport"
    elif zone_id in MANHATTAN_ZONES:
        return "manhattan"
    else:
        return "outer"

df_feat["pu_zone_type"] = df_feat["PULocationID"].apply(zone_type)
df_feat["do_zone_type"] = df_feat["DOLocationID"].apply(zone_type)

# Trip type — combines origin + destination
df_feat["trip_type"] = df_feat["pu_zone_type"] + "_to_" + df_feat["do_zone_type"]

# Is airport trip? (either end)
df_feat["is_airport_trip"] = (
    (df_feat["PULocationID"].isin(AIRPORT_ZONES)) |
    (df_feat["DOLocationID"].isin(AIRPORT_ZONES))
).astype(int)

# Same zone trip (very short, likely pickup confusion)
df_feat["same_zone"] = (
    df_feat["PULocationID"] == df_feat["DOLocationID"]
).astype(int)

print("\nZone type distribution:")
print(df_feat["trip_type"].value_counts().head(8))
print(f"\nAirport trips: {df_feat['is_airport_trip'].sum():,}")
print(f"Same zone trips: {df_feat['same_zone'].sum():,}")


Zone type distribution:
trip_type
manhattan_to_manhattan    584216
manhattan_to_outer         30603
airport_to_manhattan       22989
airport_to_outer           14974
outer_to_outer             14178
manhattan_to_airport       13419
outer_to_manhattan          6366
airport_to_airport          1175
Name: count, dtype: int64

Airport trips: 53,019
Same zone trips: 39,289


Section 3. RATECODE FEATURE 

In [ ]:

# RatecodeID tells us the pricing structure — crucial because JFK flat-rate
# trips behave completely differently from metered standard trips

ratecode_map = {
    1.0: "standard",
    2.0: "jfk",
    3.0: "newark",
    4.0: "nassau_westchester",
    5.0: "negotiated",
    6.0: "group_ride"
}
df_feat["rate_type"] = df_feat["RatecodeID"].map(ratecode_map).fillna("unknown")

print("\nRate type distribution:")
print(df_feat["rate_type"].value_counts())


Rate type distribution:
rate_type
standard              666850
jfk                    17482
negotiated              2036
newark                  1480
nassau_westchester       519
unknown                   15
Name: count, dtype: int64


Section 4. ENCODE CATEGORICAL FEATURES

In [ ]:

from sklearn.preprocessing import LabelEncoder

# Label encode low-cardinality categoricals
for col in ["store_and_fwd_flag", "pu_zone_type", "do_zone_type",
            "trip_type", "rate_type"]:
    le = LabelEncoder()
    df_feat[col + "_enc"] = le.fit_transform(df_feat[col].astype(str))
    print(f"{col}: {df_feat[col].nunique()} categories → encoded")

store_and_fwd_flag: 2 categories → encoded
pu_zone_type: 3 categories → encoded
do_zone_type: 3 categories → encoded
trip_type: 9 categories → encoded
rate_type: 6 categories → encoded


Section 5. FINAL FEATURE SET SELECTION

In [ ]:

FEATURES = [
    # Core trip features
    "trip_distance",
    "passenger_count",

    # Time features
    "pickup_hour",
    "pickup_dow",
    "pickup_month",
    "is_rush_hour",
    "is_night",
    "is_weekend",
    "season",

    # Spatial features
    "PULocationID",
    "DOLocationID",
    "is_airport_trip",
    "same_zone",
    "pu_zone_type_enc",
    "do_zone_type_enc",

    # Trip type
    "rate_type_enc",
    "store_and_fwd_flag_enc",
    "VendorID",
]

TARGET = "trip_duration"

X = df_feat[FEATURES]
y = df_feat[TARGET]

print(f"\nFinal feature matrix: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures selected ({len(FEATURES)}):")
for f in FEATURES:
    print(f"  {f}")

# Check for any remaining nulls
print(f"\nNull check — X: {X.isnull().sum().sum()}")
print(f"Null check — y: {y.isnull().sum()}")


Final feature matrix: (688382, 18)
Target shape: (688382,)

Features selected (18):
  trip_distance
  passenger_count
  pickup_hour
  pickup_dow
  pickup_month
  is_rush_hour
  is_night
  is_weekend
  season
  PULocationID
  DOLocationID
  is_airport_trip
  same_zone
  pu_zone_type_enc
  do_zone_type_enc
  rate_type_enc
  store_and_fwd_flag_enc
  VendorID

Null check — X: 0
Null check — y: 0


Section 6. SAVE THE CLEAN FEATURE SET 

In [ ]:

df_feat[FEATURES + [TARGET, "tpep_pickup_datetime", "pickup_month",
                    "month_label"]].to_parquet(
    "give the path of the folder where you have unzipped the file/Drif_detection_ds/df_2019_features.parquet", index=False
)

print("Saved: df_2019_features.parquet")
print(f"Shape: {df_feat.shape}")